In [ ]:
import cv2
import numpy as np
import json
import datetime
import os
import tempfile
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import base64
from io import BytesIO
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

# Initialize YOLOv8 model
model = YOLO("yolov8x.pt")

# Enhanced cricket object classes mapping
CRICKET_CLASSES = {
    'ball': 32,       # sports ball
    'bat': 39,        # cricket bat (cricket bat similar enough)
    'player': 0,      # person
    'stumps': 59      # chair (better approximation than bench)
}

# Cricket pitch constants (in meters)
PITCH_LENGTH = 20.12
STUMPS_HEIGHT = 0.71
STUMPS_WIDTH = 0.23
CREASE_LENGTH = 1.22  # Distance from stumps to popping crease

def estimate_3d_position(x2d, y2d, frame_width, frame_height, object_class):
    """Improved 3D position estimation with cricket-specific adjustments"""
    x_norm = x2d / frame_width
    y_norm = y2d / frame_height

    # Cricket-specific depth assumptions
    if object_class == 'ball':
        z = 0.5 + (1 - y_norm) * 3  # Higher multiplier for cricket ball trajectory
        x = (x_norm - 0.5) * 4      # Wider x-range for cricket pitch
        y = (0.5 - y_norm) * 4
    elif object_class == 'bat':
        z = 0.3 + (1 - y_norm) * 1.2
        x = (x_norm - 0.5) * 3
        y = (0.5 - y_norm) * 2
    else:  # player and stumps
        z = 0.1 + (1 - y_norm) * 0.6
        x = (x_norm - 0.5) * 3
        y = 0

    return round(x, 2), round(y, 2), round(z, 2)

def estimate_stump_positions(batsman_positions, ball_trajectory):
    """Estimate stump positions based on batsman and ball trajectory"""
    # Use the batsman position as reference point
    if not batsman_positions:
        # Default coordinates if no batsman detected
        return [
            {"x": 0.0, "y": 0.0, "z": 0.3, "t": 0.0, "estimated": True},  # Left stump
            {"x": 0.2, "y": 0.0, "z": 0.3, "t": 0.0, "estimated": True},  # Middle stump
            {"x": 0.4, "y": 0.0, "z": 0.3, "t": 0.0, "estimated": True}   # Right stump
        ]

    # Get the average batsman position
    batsman_df = pd.DataFrame(batsman_positions)
    avg_batsman = {
        "x": batsman_df['x'].mean(),
        "y": batsman_df['y'].mean(),
        "z": batsman_df['z'].mean(),
        "t": batsman_df['t'].mean()
    }

    # Determine pitch direction from ball trajectory
    pitch_direction_x = 1.0  # Default direction
    if len(ball_trajectory) > 3:
        ball_df = pd.DataFrame(ball_trajectory)

        # Try to determine pitch direction from ball movement
        if ball_df['x'].max() - ball_df['x'].min() > 0.5:  # If ball moves significantly in x
            # Calculate correlation between time and x position
            corr = ball_df['t'].corr(ball_df['x'])
            if abs(corr) > 0.3:  # If there's significant correlation
                pitch_direction_x = np.sign(corr) * -1  # Reverse direction (ball comes from bowler to batsman)

    # Estimate stumps position relative to batsman
    stump_x = avg_batsman["x"] + (pitch_direction_x * CREASE_LENGTH)

    # Create three stumps side by side
    stump_width_offset = STUMPS_WIDTH / 2
    return [
        {"x": stump_x, "y": 0.0, "z": STUMPS_HEIGHT/2, "t": avg_batsman["t"], "estimated": True},  # Middle stump
        {"x": stump_x - stump_width_offset, "y": 0.0, "z": STUMPS_HEIGHT/2, "t": avg_batsman["t"], "estimated": True},  # Left stump
        {"x": stump_x + stump_width_offset, "y": 0.0, "z": STUMPS_HEIGHT/2, "t": avg_batsman["t"], "estimated": True}   # Right stump
    ]

def create_visualization(ball_trajectory, bat_position, batsman_leg_position, unique_stumps):
    """Create visualization of the cricket scene and return it as base64 encoded image"""
    if not ball_trajectory:
        return None

    df = pd.DataFrame(ball_trajectory)
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot trajectory with time-based coloring
    sc = ax.scatter(df['x'], df['y'], df['z'], c=df['t'], cmap='viridis', s=50)
    ax.plot(df['x'], df['y'], df['z'], 'gray', alpha=0.3)  # Add trajectory line

    # Plot bat position
    ax.scatter([bat_position['x']], [bat_position['y']], [bat_position['z']],
               c='red', marker='x', s=100, label='Bat')

    # Plot batsman position
    ax.scatter([batsman_leg_position['x']], [batsman_leg_position['y']], [batsman_leg_position['z']],
               c='blue', marker='o', s=100, label='Batsman')

    # Plot stumps
    for stump in unique_stumps:
        ax.scatter([stump['x']], [stump['y']], [stump['z']],
                   c='green', marker='^', s=100)
    # Add just one label for all stumps
    if unique_stumps:
        ax.scatter([], [], [], c='green', marker='^', s=100, label='Stumps')

    ax.set_xlabel('X Position (meters)')
    ax.set_ylabel('Y Position (meters)')
    ax.set_zlabel('Z Position (meters)')
    ax.set_title('3D Cricket Scene Reconstruction')
    plt.legend()
    plt.colorbar(sc, label='Time (seconds)')

    # Save the plot to a bytes buffer
    buf = BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)

    # Encode the image as base64
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    return img_str

def process_video(video_path, skip_frames=3):
    """Process cricket video and extract trajectories and positions"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {"error": "Failed to open video file"}

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ball_trajectory = []
    bat_positions = []
    batsman_positions = []
    stump_positions = []

    frame_count = 0
    processed_frames = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Process every nth frame (better balance between speed and accuracy)
        if frame_count % skip_frames == 0:
            results = model(frame, verbose=False)[0]  # Added verbose=False to reduce output clutter
            processed_frames += 1

            for box in results.boxes:
                x1, y1, x2, y2 = map(float, box.xyxy[0])
                cls = int(box.cls)
                conf = float(box.conf)
                xc, yc = (x1 + x2) / 2, (y1 + y2) / 2

                obj_class = None
                # Adjusted confidence thresholds for cricket
                if cls == CRICKET_CLASSES['ball'] and conf > 0.15:  # Lower threshold for small ball
                    obj_class = 'ball'
                elif cls == CRICKET_CLASSES['bat'] and conf > 0.25:
                    obj_class = 'bat'
                elif cls == CRICKET_CLASSES['player'] and conf > 0.4:
                    obj_class = 'player'
                elif cls == CRICKET_CLASSES['stumps'] and conf > 0.2:
                    obj_class = 'stumps'

                if obj_class:
                    x3d, y3d, z3d = estimate_3d_position(xc, yc, frame_width, frame_height, obj_class)
                    timestamp = round(frame_count / fps, 2)

                    if obj_class == 'ball':
                        ball_trajectory.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})
                    elif obj_class == 'bat':
                        bat_positions.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})
                    elif obj_class == 'player':
                        batsman_positions.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})
                    elif obj_class == 'stumps':
                        stump_positions.append({"x": x3d, "y": y3d, "z": z3d, "t": timestamp})

        frame_count += 1

    cap.release()

    # Handle case where no batsman was detected
    batsman_leg_position = batsman_positions[-1] if batsman_positions else {"x": 0, "y": 0, "z": 0, "t": 0}

    # Fix for bat position - use the last reliable bat position or average if available
    if bat_positions:
        # Take average of last few bat positions for stability
        last_n = min(5, len(bat_positions))
        last_positions = bat_positions[-last_n:]
        bat_position = {
            "x": round(sum(pos["x"] for pos in last_positions) / last_n, 2),
            "y": round(sum(pos["y"] for pos in last_positions) / last_n, 2),
            "z": round(sum(pos["z"] for pos in last_positions) / last_n, 2),
            "t": round(sum(pos["t"] for pos in last_positions) / last_n, 2)
        }
    else:
        # If no bat detected, estimate based on batsman position
        if batsman_positions:
            bat_position = {
                "x": batsman_leg_position["x"] + 0.3,  # Slightly in front of batsman
                "y": batsman_leg_position["y"] + 0.2,
                "z": batsman_leg_position["z"] + 0.5,  # Higher than leg position
                "t": batsman_leg_position["t"]
            }
        else:
            bat_position = {"x": 0, "y": 0, "z": 0, "t": 0}

    # Improved stump detection and estimation
    unique_stumps = []
    if stump_positions:
        try:
            stump_df = pd.DataFrame(stump_positions)

            # Use DBSCAN clustering for better stump identification
            if len(stump_positions) >= 3:
                # Only cluster on x and y coordinates
                X = stump_df[['x', 'y']].values
                clustering = DBSCAN(eps=0.5, min_samples=2).fit(X)
                stump_df['cluster'] = clustering.labels_

                # Get the centroid of each cluster
                cluster_ids = set([c for c in stump_df['cluster'] if c != -1])

                # For each cluster, compute the average position
                for cluster_id in cluster_ids:
                    cluster_data = stump_df[stump_df['cluster'] == cluster_id]
                    unique_stumps.append({
                        "x": round(cluster_data['x'].mean(), 2),
                        "y": round(cluster_data['y'].mean(), 2),
                        "z": round(cluster_data['z'].mean(), 2),
                        "t": round(cluster_data['t'].mean(), 2)
                    })
            else:
                # Simple grouping for few detections
                unique_stumps = stump_df.groupby(['x', 'y']).mean().reset_index().to_dict('records')

            # Limit to 3 stumps maximum
            unique_stumps = unique_stumps[:3]

            # If we don't have enough stumps, fill in the missing ones
            if len(unique_stumps) < 3:
                estimated_stumps = estimate_stump_positions(batsman_positions, ball_trajectory)
                unique_stumps.extend(estimated_stumps[len(unique_stumps):])
        except Exception as e:
            estimated_stumps = estimate_stump_positions(batsman_positions, ball_trajectory)
            unique_stumps = estimated_stumps
    else:
        # No stumps detected - use estimation algorithm
        unique_stumps = estimate_stump_positions(batsman_positions, ball_trajectory)

    # Create visualization
    visualization_base64 = create_visualization(ball_trajectory, bat_position, batsman_leg_position, unique_stumps)

    # Create output
    output_data = {
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "video_info": {
            "filename": os.path.basename(video_path),
            "duration": round(frame_count / fps, 2),
            "frames_processed": processed_frames,
            "total_frames": frame_count
        },
        "ball_trajectory": ball_trajectory,
        "bat_position": bat_position,
        "batsman_leg_position": batsman_leg_position,
        "stump_coordinates": unique_stumps,
        "visualization": visualization_base64
    }

    return output_data

def get_module_output(video_path=None):
    """Main function to be called by the API endpoint"""
    if video_path is None:
        return {
            "status": "error",
            "message": "No video file provided"
        }

    try:
        results = process_video(video_path)
        return {
            "status": "success",
            "message": "Video processing completed successfully",
            "data": results
        }
    except Exception as e:
        return {
            "status": "error",
            "message": f"Error processing video: {str(e)}"
        }